# 04_labeling.ipynb - Fase 4: Triple Barrier Labeling

**Objetivo**: Generar un target binario para ML: {1: TP, 0: SL o Timeout}

La lógica de triple barrier sigue siendo la misma para decidir el resultado; el output usado por el modelo se binariza para simplificar el entrenamiento.

In [1]:
import sys
import pandas as pd
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from config.settings import DATA, FEATURES, RISK, BACKTEST
from src.labeling import create_labeled_dataset

print("Imports listos")

Imports listos


## Labeling para cada ticker admitido

In [2]:
features_dir = REPO_ROOT / "data" / "features"
labeled_dir = REPO_ROOT / "data" / "labeled"
labeled_dir.mkdir(parents=True, exist_ok=True)

admitted_df = pd.read_csv(REPO_ROOT / "data" / "universe_admitted.csv")

for _, row in admitted_df.iterrows():
    ticker = row['ticker']
    csl = row['csl']
    print(f"\nProcesando {ticker} (Csl={csl:.3f})")

    feature_path = features_dir / f"features_{ticker}.csv"
    if feature_path.exists():
        df = pd.read_csv(feature_path, parse_dates=['Datetime'])
        print(f"  Cargado features desde {feature_path}")
    else:
        print(f"  ⚠️ Features no encontrados en {feature_path}")
        continue

    if df.empty:
        print("  ❌ No hay datos disponibles")
        continue

    labeled = create_labeled_dataset(
        df,
        csl,
        tp_sl_ratio=BACKTEST.tp_sl_ratio,
        max_bars=BACKTEST.max_bars,
        binary_target=True,
    )

    if labeled.empty:
        print("  ⚠️ No se generaron etiquetas para este ticker")
        continue

    print(f"  ✅ Muestras etiquetadas: {len(labeled)}")
    print(f"  Distribución: {labeled['label'].value_counts().to_dict()}")

    output_path = labeled_dir / f"labeled_{ticker}.csv"
    labeled.to_csv(output_path, index=False)
    print(f"  Guardado: {output_path}")


Procesando MCRB (Csl=2.065)
  Cargado features desde /home/paolo/Escritorio/projecto_machine_learning_quant/smallcap-quant-ml/data/features/features_MCRB.csv
  ✅ Muestras etiquetadas: 3637
  Distribución: {0: 2461, 1: 1176}
  Guardado: /home/paolo/Escritorio/projecto_machine_learning_quant/smallcap-quant-ml/data/labeled/labeled_MCRB.csv

Procesando ABSI (Csl=2.243)
  Cargado features desde /home/paolo/Escritorio/projecto_machine_learning_quant/smallcap-quant-ml/data/features/features_ABSI.csv
  ✅ Muestras etiquetadas: 5032
  Distribución: {0: 2953, 1: 2079}
  Guardado: /home/paolo/Escritorio/projecto_machine_learning_quant/smallcap-quant-ml/data/labeled/labeled_ABSI.csv

Procesando IMUX (Csl=2.355)
  Cargado features desde /home/paolo/Escritorio/projecto_machine_learning_quant/smallcap-quant-ml/data/features/features_IMUX.csv
  ✅ Muestras etiquetadas: 4997
  Distribución: {0: 3149, 1: 1848}
  Guardado: /home/paolo/Escritorio/projecto_machine_learning_quant/smallcap-quant-ml/data/label